<a href="https://colab.research.google.com/github/programminghistorian/ph-submissions/blob/gh-pages/assets/enablar-lesson-5/enablar-lesson-5b-analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploring Library Catalogues as Data
## Part II. Data analysis and visualisation
### Import the necessary libraries

In [ ]:
%pip install pandas matplotlib matplotlib-venn numpy

In [36]:
import pandas as pd
import re
import numpy as np
import os
import urllib

In [ ]:
csv_files = ['bib_20250706_full_000_00.csv', 'bib_20250706_full_000_01.csv']
base_url = 'https://github.com/programminghistorian/ph-submissions/raw/refs/heads/gh-pages/assets/enablar-lesson-5/'
for csv_file in csv_files:
    if not os.path.exists(csv_file):
        urllib.request.urlretrieve(base_url + csv_file, csv_file)

### Import from the previously created CSV files

In [ ]:
df_a = pd.read_csv('bib_20250706_full_000_00.csv')
df_b = pd.read_csv('bib_20250706_full_000_01.csv')

print(f'Catalogue A: {len(df_a)} records')
print(f'Catalogue B: {len(df_b)} records')

In [ ]:
df_a.head()

### Asking a question of the collection

LCSH's [approved changes](https://classweb.org/approved-subjects/)

In [ ]:
lc_changes = ['McKinley, Mount', 'Enslaved persons', 'Noncitizen']

for change in lc_changes:
    count = df_a['subjects'].str.contains(change, na=False).sum()
    print(f'{change}: {count} records')

### Comparing two datasets

#### Function 2: `headings_matching`

In [ ]:
def headings_matching(df, keyword):
    """Return the set of subject headings used on records whose
    subjects column contains the keyword."""

    matching = df[df['subjects'].str.contains(keyword, na=False)]

    headings = set()
    for subjects_str in matching['subjects']:
        headings.update(subjects_str.split('|'))

    return headings

In [ ]:
headings_a = headings_matching(df_a, 'Immigra')
headings_b = headings_matching(df_b, 'Immigra')

print(f'Catalogue A uses {len(headings_a)} distinct headings on immigration records')
print(f'Catalogue B uses {len(headings_b)} distinct headings on immigration records')

#### Function 3: `compare_sets`

In [ ]:
def compare_sets(set_x, set_y):
    """Compare two sets. Return a dict with three keys:
    shared (in both), only_in_x, and only_in_y."""

    return {
        'shared':    set_x & set_y,
        'only_in_x': set_x - set_y,
        'only_in_y': set_y - set_x,
    }

In [ ]:
result = compare_sets(headings_a, headings_b)

print(f'Shared headings:     {len(result["shared"])}')
print(f'Only in catalogue A: {len(result["only_in_x"])}')
print(f'Only in catalogue B: {len(result["only_in_y"])}')

print('\nSample headings only in catalogue A:')
for h in sorted(result['only_in_x'])[:5]:
    print(f'  - {h}')

### Visualization: Creating a Venn diagram

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import textwrap

In [ ]:
venn2([headings_a, headings_b], ('Catalogue 1', 'Catalogue 2'))
plt.show()
plt.close()

Create an annotated Venn diagram:

In [ ]:
venn_diagram = venn2([headings_a, headings_b], ('Catalogue 1', 'Catalogue 2'))
plt.annotate(
    text=textwrap.fill(', '.join(result["shared"]), width=90),
    xy=venn_diagram.get_label_by_id('11').get_position() - np.array([0, 0.05]),
    xytext=(-150,-150),
    ha='left',
    textcoords='offset points',
    bbox=dict(
        boxstyle='round,pad=0.5',
        fc='gray',
        alpha=0.1),
    arrowprops=dict(
        arrowstyle='->',
        connectionstyle='arc3,rad=0.5',
        color='gray'
    )
)
plt.show()
plt.close()

see API documentations:

* matplotlib's [annotate()](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.annotate.html#matplotlib.axes.Axes.annotate) function
* textwrap's [fill()](https://docs.python.org/3/library/textwrap.html#textwrap.fill) function

Display the subjects of Catalogue 1, but removing their closing dots.

In [ ]:
def format_subjects(subjects):
    # regex to find dot at the end of the string after small case letter
    subject_pattern = re.compile(r'[a-z]\.$')
    # remove these dots
    subjects = [subject_pattern.sub('', s) for s in subjects]
    # join subjects
    subjects_str = ', '.join(subjects)
    # fill into 70 char long lines
    return textwrap.fill(subjects_str, width=65)

In [ ]:
venn_diagram = venn2([headings_a, headings_b], ('Catalogue 1', 'Catalogue 2'))
plt.annotate(
    text=format_subjects(result["only_in_x"]),
    # 10 denotes the first set (Catalogue 1)
    xy=venn_diagram.get_label_by_id('10').get_position() - np.array([-0.05, 0]), # adjust position of the arrow
    # adjust the position of the text box
    xytext=(-80,-220),
    ha='left',
    textcoords='offset points',
    bbox=dict(
        boxstyle='round,pad=0.5',
        fc='gray',
        alpha=0.1),
    arrowprops=dict(
        arrowstyle='->',
        connectionstyle='arc3,rad=0.5',
        color='gray'
    )
)
plt.show()
plt.close()